# Summarize an EULA

Run this notebook from the repository root after following the README setup. Change `INPUT_FILE` and `SENTENCE_COUNT`, then run all cells. Source files belong in `data/` and generated summaries are saved in `outputs/`.

In [ ]:
# Uncomment and run each line once if the package is missing.
# %pip install jupyterlab
# %pip install networkx
# %pip install numpy
# %pip install pandas
# %pip install scikit-learn

In [ ]:
import re
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

INPUT_FILE = Path("data/ApacheEULA.csv")
OUTPUT_FILE = Path("outputs/eula_summary.txt")
TEXT_COLUMN = "Description"
SENTENCE_COUNT = 10

def split_sentences(text):
    return [sentence.strip() for sentence in re.split(r"(?<=[.!?])\s+", text) if sentence.strip()]


def read_document(input_path, text_column):
    if input_path.suffix.lower() == ".txt":
        return input_path.read_text(encoding="utf-8")
    if input_path.suffix.lower() != ".csv":
        raise ValueError("INPUT_FILE must be a .csv or .txt file.")

    dataframe = pd.read_csv(input_path)
    if text_column not in dataframe.columns:
        raise ValueError(f"Column '{text_column}' was not found.")
    return " ".join(dataframe[text_column].dropna().astype(str))


def summarize_text(text, sentence_count):
    sentences = split_sentences(text)
    if sentence_count < 1:
        raise ValueError("SENTENCE_COUNT must be at least 1.")
    if len(sentences) <= sentence_count:
        return sentences

    vectors = TfidfVectorizer(stop_words="english").fit_transform(sentences)
    similarity = cosine_similarity(vectors)
    np.fill_diagonal(similarity, 0)
    scores = nx.pagerank(nx.from_numpy_array(similarity))
    best_indexes = sorted(sorted(scores, key=scores.get, reverse=True)[:sentence_count])
    return [sentences[index] for index in best_indexes]


def summarize_file(input_path, sentence_count, text_column):
    return summarize_text(read_document(input_path, text_column), sentence_count)


In [ ]:
summary = summarize_file(INPUT_FILE, SENTENCE_COUNT, TEXT_COLUMN)
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE.write_text("\n\n".join(summary) + "\n", encoding="utf-8")

print(f"Saved {len(summary)} sentences to {OUTPUT_FILE}")
print("\n".join(f"{index}. {sentence}" for index, sentence in enumerate(summary, start=1)))